# Deep Learning project

Team members:

* Rimsha Afzal,
* Nika Dariani,
* Irina Krylova, 255809

## Setup

Run this notebook from the project root. In Colab, upload or mount the project folder first, then set `PROJECT_ROOT` to that folder.

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Could not find the project root (a folder containing src/). Started from: " + str(Path.cwd()))

sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [4]:
from baseline_retrieval import run_baseline
from embedding_io import load_embeddings

## Paths

The subset query file below is the one compatible with the copied test embeddings. The larger subset query file uses a different index space, so it is not used here.

In [2]:
QUERY_JSON = PROJECT_ROOT / "data/celeba_subset/queries/test_embedding_celeba_evaluation.json"
IMAGE_EMBEDDING_DIR = PROJECT_ROOT / "data/celeba_subset/embeddings/test"
TEXT_EMBEDDING_DIR = PROJECT_ROOT / "data/celeba_subset/embeddings"
CHECK_JSON = PROJECT_ROOT / "data/celeba_subset/checks/query_embedding_compatibility_check.json"
SUMMARY_CSV = PROJECT_ROOT / "outputs/baseline_run/summary.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs/baseline_run/notebook_test_subset_alpha2.00_beta1.00"

for path in [QUERY_JSON, IMAGE_EMBEDDING_DIR, TEXT_EMBEDDING_DIR, CHECK_JSON]:
    if not path.exists():
        raise FileNotFoundError(path)

## Data Sanity Check

Before running retrieval, check that the query file, image embeddings, and text embeddings refer to the same subset.

In [5]:
with QUERY_JSON.open("r", encoding="utf-8") as handle:
    query_entries = json.load(handle)

with CHECK_JSON.open("r", encoding="utf-8") as handle:
    compatibility_check = json.load(handle)

image_embeddings, image_ids = load_embeddings(IMAGE_EMBEDDING_DIR, "image_embeddings", "cpu")
text_embeddings, text_ids = load_embeddings(TEXT_EMBEDDING_DIR, "text_embeddings", "cpu")

query_instances = sum(len(entry["ground_truth"]) for entry in query_entries)
target_links = sum(
    len(targets)
    for entry in query_entries
    for targets in entry["ground_truth"].values()
)

print("compatibility check:", compatibility_check["status"])
print("query types:", len(query_entries))
print("query instances:", query_instances)
print("target links:", target_links)
print("image embeddings:", tuple(image_embeddings.shape), "ids:", len(image_ids))
print("text embeddings:", tuple(text_embeddings.shape), "prompts:", len(text_ids))

compatibility check: pass
query types: 13
query instances: 1640
target links: 26829
image embeddings: (8035, 512) ids: 8035
text embeddings: (40, 512) prompts: 40


# 1. Introduction

# 2. Related work

# 3. Method

[a detailed, formal overview of the solution you developed.
This must include a mathematical description of your architecture, the forward pass, and
the loss functions governing the training process (if applicable). Clearly highlight your
original contributions and adequately cite relevant literature.]

## 3.1. Baseline

The retrieval query is constructed by latent-space arithmetic over L2-normalized CLIP embeddings:

$$
\mathbf{q} = \frac{\tilde{\mathbf{q}}}{\lVert \tilde{\mathbf{q}} \rVert_2},
\qquad
\tilde{\mathbf{q}} = \mathbf{v}_{\text{ref}}
+ \alpha \sum_{p \in \mathcal{P}} \mathbf{t}_p
- \beta \sum_{n \in \mathcal{N}} \mathbf{t}_n,
\qquad \alpha = \beta = 1,
$$

where $\mathbf{v}_{\text{ref}} \in \mathbb{R}^{512}$ is the CLIP image embedding of the reference image; $\mathcal{P}$ and $\mathcal{N}$ are the sets of positive and negative attributes in the query (e.g. $+\textit{Smiling} \in \mathcal{P}$, $-\textit{Eyeglasses} \in \mathcal{N}$); and $\mathbf{t}_p, \mathbf{t}_n$ are the corresponding CLIP text embeddings. All embeddings are L2-normalized, so retrieval ranks the gallery by cosine similarity, computed as a dot product.

## 3.2. Gated fusion grid search

The gated variant generalizes the baseline by re-weighting the contributions, keeping the same form but treating $\alpha$ (positive conditions) and $\beta$ (negative conditions) as tunable gates rather than fixing them to $1$:

$$
\mathbf{q} = \frac{\tilde{\mathbf{q}}}{\lVert \tilde{\mathbf{q}} \rVert_2},
\qquad
\tilde{\mathbf{q}} = \gamma\,\mathbf{v}_{\text{ref}}
+ \alpha \sum_{p \in \mathcal{P}} \mathbf{t}_p
- \beta \sum_{n \in \mathcal{N}} \mathbf{t}_n.
$$

Since the final L2-normalization makes the query invariant to a global rescaling of $\tilde{\mathbf{q}}$, only the ratios of the gates are identifiable; we therefore fix the reference-image gate $\gamma = 1$ and select $(\alpha, \beta)$ by grid search, retaining the pair that maximizes Recall@$1$ on the validation split. The baseline is recovered as the special case $\gamma = \alpha = \beta = 1$.

## 3.3. Visual directions

Why it helps? Because text and visual directions live in separate "cones" in the embedding space (SOURCE: CLAY paper, Liang et al. 2022), which is called a modality gap

Liang et al 2022: "They show image and text embeddings in CLIP are "embedded at arm's length" in two completely separate regions of the unit sphere. The Euclidean distance between the image and text embedding clusters for pretrained CLIP is 0.82, and they show this is near the contrastive-loss optimum — in their words, "the default gap distance ‖Δgap‖=0.82 actually achieves the global minimum, and shifting toward closing the gap increases the contrastive loss" — so the gap is preserved by training rather than incidental. The gap originates in the cone effect: at initialization a deep network's outputs occupy a narrow cone. They measure average pairwise cosines of "0.56, 0.47, 0.51 respectively for the 3 models" (ResNet, ViT, Text Transformer), with minimum cosines of only "0.23, 0.05, 0.01" — i.e., embeddings are far from spanning the sphere. This is the root cause of why your text-derived directions and v_ref barely interact in a metrically meaningful way: adding a text vector moves the query partly along the "gap" axis rather than along a semantically discriminative direction inside the image cone."

If we use visual directions, then the signal from the attribute stays in the same cone

We replace the CLIP text embeddings with *visual* attribute directions. For each CelebA attribute $a$, the direction is estimated on the train split as the difference between the mean image embedding of positive and negative examples:

$$
\mathbf{d}_a = \frac{\boldsymbol{\delta}_a}{\lVert \boldsymbol{\delta}_a \rVert_2},
\qquad
\boldsymbol{\delta}_a =
\frac{1}{|\mathcal{I}_a^{+}|}\sum_{i \in \mathcal{I}_a^{+}} \mathbf{v}_i
-
\frac{1}{|\mathcal{I}_a^{-}|}\sum_{i \in \mathcal{I}_a^{-}} \mathbf{v}_i,
$$

where $\mathcal{I}_a^{+}$ and $\mathcal{I}_a^{-}$ index the train images with attribute $a$ present ($+1$) and absent ($-1$), and $\mathbf{v}_i$ is the CLIP image embedding of image $i$. The query is then formed from the reference image and the signed visual directions, with $\alpha$ gating the positive directions and $\beta$ the negative ones:

$$
\mathbf{q} = \frac{\tilde{\mathbf{q}}}{\lVert \tilde{\mathbf{q}} \rVert_2},
\qquad
\tilde{\mathbf{q}} = \gamma\,\mathbf{v}_{\text{ref}}
+ \alpha \sum_{p \in \mathcal{P}} \mathbf{d}_p
- \beta \sum_{n \in \mathcal{N}} \mathbf{d}_n.
$$

Note that each single-attribute direction $\mathbf{d}_a$ is *itself* a difference of positive and negative image means, so the per-attribute subtraction is baked into $\mathbf{d}_a$ and is independent of the $+/-$ sign carried by the query.

## 3.4. Hybrid text and visual fusion

The hybrid variant combines the reference image with both the text and visual condition directions. Following the convention that $\alpha$ gates positive conditions and $\beta$ gates negative conditions, the positive side splits into a text gate $\alpha_{\text{txt}}$ and a visual gate $\alpha_{\text{vis}}$, while the visual negatives are gated by $\beta_{\text{vis}}$:

$$
\mathbf{q} = \frac{\tilde{\mathbf{q}}}{\lVert \tilde{\mathbf{q}} \rVert_2},
\qquad
\tilde{\mathbf{q}} = \gamma\,\mathbf{v}_{\text{ref}}
+ \alpha_{\text{txt}}\,\boldsymbol{\Delta}_{\text{txt}}
+ \alpha_{\text{vis}}\,\boldsymbol{\Delta}_{\text{vis}}^{+}
- \beta_{\text{vis}}\,\boldsymbol{\Delta}_{\text{vis}}^{-},
$$

with the aggregated condition vectors

$$
\boldsymbol{\Delta}_{\text{txt}} = \sum_{p \in \mathcal{P}} \mathbf{t}_p - \sum_{n \in \mathcal{N}} \mathbf{t}_n,
\qquad
\boldsymbol{\Delta}_{\text{vis}}^{+} = \sum_{p \in \mathcal{P}} \mathbf{d}_p,
\qquad
\boldsymbol{\Delta}_{\text{vis}}^{-} = \sum_{n \in \mathcal{N}} \mathbf{d}_n.
$$

Here $\boldsymbol{\Delta}_{\text{txt}}$ already carries its own internal positive/negative split; the text contribution is therefore gated as a single positive-side term by $\alpha_{\text{txt}}$.

### 3.4.1. Hybrid text and visual fusion with prompt ensembling

Prompt averaging: Radford et al, 2021 (CLIP paper, they averaged 80 prompts)

Prompt difference: Patashnik et al, 2021. "The "subtract a baseline" construction's actual published precedent is StyleCLIP (Patashnik et al., "StyleCLIP: Text-Driven Manipulation of StyleGAN Imagery," ICCV 2021), the "global directions" method: they build a direction as Δt = normalize(ensemble(target prompts) − ensemble(neutral prompts)) — e.g. "a sports car" minus "a car" — averaged over the CLIP prompt templates."

Prompt ensembling modifies only how each per-attribute text vector $\mathbf{t}_a$ entering $\boldsymbol{\Delta}_{\text{txt}}$ is computed; the fusion formulas above are unchanged. Let $c_1, \dots, c_{12}$ denote the $12$ carrier templates and $\text{embed}(\cdot)$ the CLIP text encoder. We evaluate two constructions.

*Ensemble of positive prompts* — the mean of the $12$ positive-prompt embeddings:

$$
\mathbf{t}_a^{\text{ens}} = \frac{\boldsymbol{e}_a}{\lVert \boldsymbol{e}_a \rVert_2},
\qquad
\boldsymbol{e}_a = \frac{1}{12} \sum_{i=1}^{12} \text{embed}\!\left(c_i(a)\right).
$$

For the ensemble of positive prompts, the following prompts phrases were used: "a photo of a person {phrase}", "a close-up photo of a person {phrase}", "a headshot of a person {phrase}", "a portrait of someone {phrase}", "a cropped photo of a person {phrase}", "a good photo of a person {phrase}", "a bad photo of a person {phrase}", "a photo of the face of a person {phrase}", "a photo of a celebrity {phrase}", "a high quality photo of a person {phrase}", "a low quality photo of a person {phrase}", "an image of a person {phrase}". Instead of {phrase}, 40 CelebA attributes were inserted, transformed (e.g. "5_o_Clock_Shadow" --> "with a five o'clock shadow")

*Difference ensemble* — the normalized difference between the mean positive-prompt embedding and the mean neutral-prompt embedding:

$$
\mathbf{t}_a^{\text{diff}} = \frac{\boldsymbol{e}_a^{+} - \boldsymbol{e}_a^{0}}{\lVert \boldsymbol{e}_a^{+} - \boldsymbol{e}_a^{0} \rVert_2},
\qquad
\boldsymbol{e}_a^{+} = \frac{1}{12}\sum_{i=1}^{12}\text{embed}\!\left(c_i^{+}(a)\right),
\quad
\boldsymbol{e}_a^{0} = \frac{1}{12}\sum_{i=1}^{12}\text{embed}\!\left(c_i^{0}\right),
$$

where $c_i^{+}(a)$ slots attribute $a$ into carrier $i$ and $c_i^{0}$ is the corresponding neutral (attribute-free) carrier. The neutral carriers contain no negation; this avoids CLIP's poor handling of negated text, which would otherwise make $\boldsymbol{e}_a^{+} - \boldsymbol{e}_a^{0}$ collapse toward zero.

For the difference ensemble, along the aforementioned 12 prompts, the following neutral prompts were used: "a photo of a person's face.", "a close-up photo of a person's face.", "a headshot of a person.", "a portrait of a celebrity.", "a cropped photo of a person's face.", "a good photo of a person's face.", "a bad photo of a person's face.", "a photo of the face of a person.", "a photo of a celebrity.", "a high quality photo of a person's face.", "a low quality photo of a person's face.", "an image of a person's face."

## 3.5. Reliability weighted hybrid text and visual fusion

Query difficulty score based on images present in data

### 3.5.1. Correlation-based reliability weighted hybrid text and visual fusion

# 4. Experiments and results

Ran experiments on a local subset of N validation and M test images.

For each method, the grid search is run.

[a rigorous description of the training and evaluation strategy. Exten
sively motivate your methodological choices, including network capacity, optimizer selec
tion, hyperparameter tuning, and data sampling strategies]

i suggest to put here also: [an extensive presentation of your findings. You must report stan
dard retrieval metrics (Recall@K). Organize your scores in comparative tables, and in
clude charts depicting learning curves (if applicable), qualitative retrieval examples (suc
cesses and failure cases), and any other visual representations that aid in understanding the
model’s behavior]

## 4.1. Baseline

In [6]:
from typing import Any

import torch.nn.functional as F

# embeddings helper functions
def l2_normalize(embeddings: torch.Tensor) -> torch.Tensor:
    """Return embeddings scaled to unit length, row by row."""
    single = embeddings.dim() == 1
    if single:
        embeddings = embeddings.unsqueeze(0)
    normalized = F.normalize(embeddings.float(), p=2, dim=1, eps=1e-12)
    return normalized.squeeze(0) if single else normalized

def load_embeddings(
    output_dir: str | Path,
    name: str,
    device: str = "cpu",
) -> tuple[torch.Tensor, list[Any]]:
    """Load a saved embedding tensor and its matching row IDs."""
    output_dir = Path(output_dir)
    embeddings_path = output_dir / f"{name}.pt"
    ids_path = output_dir / f"{name}_ids.npy"

    if not embeddings_path.exists():
        raise FileNotFoundError(f"Missing embeddings file: {embeddings_path}")
    if not ids_path.exists():
        raise FileNotFoundError(f"Missing embedding IDs file: {ids_path}")

    embeddings = torch.load(embeddings_path, map_location=device, weights_only=True)
    ids = np.load(ids_path, allow_pickle=True).tolist()
    return embeddings, ids

In [7]:
from typing import Iterable

# evaluation helper functions
def recall_at_k(
    retrieved_indices: list[int],
    ground_truth_indices: set[int],
    k: int,
) -> float:
    """Return 1.0 if a valid target appears in the top K results."""
    if k <= 0:
        raise ValueError("k must be positive.")

    top_k = set(retrieved_indices[:k])
    return 1.0 if top_k & ground_truth_indices else 0.0


def precision_at_k(
    retrieved_indices: list[int],
    ground_truth_indices: set[int],
    k: int,
) -> float:
    """Return the fraction of top K results that are valid targets."""
    if k <= 0:
        raise ValueError("k must be positive.")

    top_k = retrieved_indices[:k]
    if len(top_k) == 0:
        return 0.0

    hits = sum(1 for idx in top_k if idx in ground_truth_indices)
    return hits / k


def evaluate_single_ranking(
    retrieved_indices: list[int],
    ground_truth_indices: Iterable[int],
    ks: tuple[int, ...] = (1, 5, 10),
) -> dict[str, float]:
    """Compute Recall@K and Precision@K for one source image."""
    ground_truth_set = set(int(idx) for idx in ground_truth_indices)
    metrics = {}

    for k in ks:
        metrics[f"recall@{k}"] = recall_at_k(
            retrieved_indices=retrieved_indices,
            ground_truth_indices=ground_truth_set,
            k=k,
        )
        metrics[f"precision@{k}"] = precision_at_k(
            retrieved_indices=retrieved_indices,
            ground_truth_indices=ground_truth_set,
            k=k,
        )

    return metrics

def average_metrics(metric_rows: list[dict[str, float]]) -> dict[str, float]:
    """Average metric dictionaries across source images."""
    if len(metric_rows) == 0:
        raise ValueError("Cannot average an empty list of metric rows.")

    averaged = {}
    for name in metric_rows[0].keys():
        averaged[name] = sum(row[name] for row in metric_rows) / len(metric_rows)

    return averaged

In [14]:
import csv
from typing import Any

# baseline retrieval helper functions
def parse_query_string(query: str) -> tuple[list[str], list[str]]:
    """Split a signed query string into positive and negative attributes."""
    positive = []
    negative = []
    for part in query.split(","):
        part = part.strip()
        if part.startswith("+"):
            positive.append(part[1:].strip())
        elif part.startswith("-"):
            negative.append(part[1:].strip())
    return positive, negative


def attribute_to_prompt(attribute: str) -> str:
    """Convert a CelebA attribute name into the saved CLIP text prompt."""
    return f"a photo of a person who is {attribute.replace('_', ' ').lower()}"


def build_query_embeddings(
    reference_embeddings: torch.Tensor,
    positive_embeddings: torch.Tensor | None,
    negative_embeddings: torch.Tensor | None,
    alpha: float = 1.0,
    beta: float = 1.0,
) -> torch.Tensor:
    """Apply image + positive text - negative text arithmetic fusion."""
    queries = reference_embeddings.float()
    if positive_embeddings is not None and positive_embeddings.numel() > 0:
        queries = queries + alpha * positive_embeddings.float().sum(dim=0, keepdim=True)
    if negative_embeddings is not None and negative_embeddings.numel() > 0:
        queries = queries - beta * negative_embeddings.float().sum(dim=0, keepdim=True)
    return l2_normalize(queries)


def retrieve_top_k_batch(
    query_embeddings: torch.Tensor,
    image_embeddings: torch.Tensor,
    image_ids: list[int],
    k: int,
    exclude_ids: list[int],
    device: str = "cpu",
) -> list[list[tuple[int, float]]]:
    """Retrieve top K image IDs for each query embedding."""
    query_embeddings = query_embeddings.to(device).float()
    image_embeddings = image_embeddings.to(device).float()
    similarities = query_embeddings @ image_embeddings.T

    id_to_row = {int(image_id): row for row, image_id in enumerate(image_ids)}
    for query_row, image_id in enumerate(exclude_ids):
        row = id_to_row.get(int(image_id))
        if row is not None:
            similarities[query_row, row] = -torch.inf

    values, indices = torch.topk(similarities, k=k, dim=1)
    results = []
    for row in range(query_embeddings.shape[0]):
        results.append(
            [
                (int(image_ids[index]), float(score))
                for index, score in zip(indices[row].cpu().tolist(), values[row].cpu())
            ]
        )
    return results


def load_ground_truth(path: str | Path) -> list[dict[str, Any]]:
    """Load a non-empty query/ground-truth JSON file."""
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        raise FileNotFoundError(f"Missing or empty ground-truth file: {path}")
    with path.open("r", encoding="utf-8") as handle:
        data = json.load(handle)
    if not isinstance(data, list):
        raise ValueError("Ground-truth JSON must contain a list of query records.")
    return data


def run_baseline(
    query_json: str | Path,
    image_embedding_dir: str | Path,
    text_embedding_dir: str | Path,
    output_dir: str | Path | None = None,
    alpha: float = 1.0,
    beta: float = 1.0,
    top_k: tuple[int, ...] = (1, 5, 10),
    batch_size: int = 256,
    device: str = "cpu",
    save_metrics: bool = True,
    save_predictions: bool = False,
    save_run_config: bool = False,
) -> dict[str, Any]:
    """Run the arithmetic baseline, optionally saving metrics/predictions/config to output_dir."""
    if save_metrics or save_predictions or save_run_config:
        if output_dir is None:
            raise ValueError("output_dir is required when save_metrics, save_predictions, or save_run_config is True.")
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

    image_embeddings, image_ids = load_embeddings(image_embedding_dir, "image_embeddings", "cpu")
    text_embeddings, text_ids = load_embeddings(text_embedding_dir, "text_embeddings", "cpu")
    image_ids = [int(image_id) for image_id in image_ids]
    image_row = {image_id: row for row, image_id in enumerate(image_ids)}
    text_lookup = {str(prompt): text_embeddings[row] for row, prompt in enumerate(text_ids)}
    entries = load_ground_truth(query_json)

    max_k = max(top_k)
    metric_rows = []
    prediction_rows = [] if save_predictions else None

    for entry in entries:
        positive, negative = parse_query_string(entry["query"])
        positive_embeddings = torch.stack(
            [text_lookup[attribute_to_prompt(attribute)] for attribute in positive]
        ) if positive else None
        negative_embeddings = torch.stack(
            [text_lookup[attribute_to_prompt(attribute)] for attribute in negative]
        ) if negative else None

        reference_items = [(int(reference), targets) for reference, targets in entry["ground_truth"].items()]
        for start in range(0, len(reference_items), batch_size):
            batch = reference_items[start:start + batch_size]
            references = [reference for reference, _ in batch]
            reference_rows = [image_row[reference] for reference in references]
            reference_embeddings = image_embeddings[reference_rows]
            query_embeddings = build_query_embeddings(
                reference_embeddings,
                positive_embeddings,
                negative_embeddings,
                alpha=alpha,
                beta=beta,
            )
            rankings = retrieve_top_k_batch(
                query_embeddings,
                image_embeddings,
                image_ids,
                k=max_k,
                exclude_ids=references,
                device=device,
            )

            for (reference, targets), ranking in zip(batch, rankings):
                retrieved = [image_id for image_id, _ in ranking]
                metrics = evaluate_single_ranking(retrieved, targets, ks=top_k)
                metric_rows.append(metrics)

                if prediction_rows is not None:
                    scores = [score for _, score in ranking]
                    prediction_rows.append(
                        {
                            "query": entry["query"],
                            "reference_index": reference,
                            "target_indices": " ".join(str(int(target)) for target in targets),
                            "retrieved_indices": " ".join(str(index) for index in retrieved),
                            "similarity_scores": " ".join(f"{score:.6f}" for score in scores),
                            **metrics,
                        }
                    )

    metrics = average_metrics(metric_rows)
    result = {
        "method": "clip_arithmetic_baseline",
        "query_json": str(query_json),
        "image_embedding_dir": str(image_embedding_dir),
        "text_embedding_dir": str(text_embedding_dir),
        "alpha": alpha,
        "beta": beta,
        "top_k": list(top_k),
        "query_instances": len(metric_rows),
        "gallery_size": len(image_ids),
        "metrics": metrics,
    }

    if save_metrics:
        (output_dir / "metrics.json").write_text(json.dumps(result, indent=2) + "\n", encoding="utf-8")
    if prediction_rows is not None:
        with (output_dir / "predictions.csv").open("w", encoding="utf-8", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=list(prediction_rows[0].keys()))
            writer.writeheader()
            writer.writerows(prediction_rows)
    if save_run_config:
        (output_dir / "run_config.json").write_text(json.dumps(result, indent=2) + "\n", encoding="utf-8")
    return result

In [15]:
# scripts/run_baseline.py adapted for the notebook: each CLI --flag becomes a
# plain variable holding the same default, then a direct call to run_baseline()
# in place of parse_args()/main(). Paths are relative to data/, independent of
# PROJECT_ROOT/src.

baseline_query_json = PROJECT_ROOT / "data/celeba_subset/queries/test_embedding_celeba_evaluation.json"
baseline_image_embedding_dir = PROJECT_ROOT / "data/celeba_subset/embeddings/test"
baseline_text_embedding_dir = PROJECT_ROOT / "data/celeba_subset/embeddings"

alpha = 1.0
beta = 1.0
batch_size = 256
device = "cpu"
save_predictions = False
save_run_config = False

baseline_output_dir = PROJECT_ROOT / "tmp_outputs/baseline_run" / f"test_subset_alpha{alpha:.2f}_beta{beta:.2f}"

baseline_result = run_baseline(
    query_json=baseline_query_json,
    image_embedding_dir=baseline_image_embedding_dir,
    text_embedding_dir=baseline_text_embedding_dir,
    output_dir=baseline_output_dir,
    alpha=alpha,
    beta=beta,
    batch_size=batch_size,
    device=device,
    save_predictions=save_predictions,
    save_run_config=save_run_config,
)

pd.DataFrame([baseline_result["metrics"]])

,recall@1,precision@1,recall@5,precision@5,recall@10,precision@10
0,0.05,0.05,0.112195,0.031707,0.170732,0.026098


## 4.2. Gated fusion grid search

grid: alpha=[0.25, 0.5, 1, 1.5, 2], beta=[0.25, 0.5, 1, 1.5, 2]

The best setting from the small alpha/beta sweep was `alpha=2.0`, `beta=1.0`.
This cell saves only `metrics.json` to keep experiment folders small.

In [18]:
import itertools

alphas = [0.25, 0.5, 1.0, 1.5, 2.0]
betas = [0.25, 0.5, 1.0, 1.5, 2.0]

grid_results = []
for grid_alpha, grid_beta in itertools.product(alphas, betas):
    grid_result = run_baseline(
        query_json=baseline_query_json,
        image_embedding_dir=baseline_image_embedding_dir,
        text_embedding_dir=baseline_text_embedding_dir,
        alpha=grid_alpha,
        beta=grid_beta,
        batch_size=batch_size,
        device=device,
        save_metrics=False,
        save_predictions=False,
        save_run_config=False,
    )
    grid_results.append({"alpha": grid_alpha, "beta": grid_beta, **grid_result["metrics"]})

grid_summary = pd.DataFrame(grid_results)
grid_summary.sort_values("recall@10", ascending=False).reset_index(drop=True).head(5)

,alpha,beta,recall@1,precision@1,recall@5,precision@5,recall@10,precision@10
0,2.0,1.50,0.059756,0.059756,0.150000,0.039268,0.228659,0.033659
1,2.0,0.50,0.055488,0.055488,0.148780,0.038780,0.226829,0.033232
2,2.0,2.00,0.059756,0.059756,0.151220,0.039634,0.226829,0.033720
3,2.0,1.00,0.056098,0.056098,0.150000,0.038780,0.226220,0.033232
4,2.0,0.25,0.054878,0.054878,0.146341,0.038049,0.225000,0.032988


### Alpha/Beta Sweep Summary

The sweep below compares different text-direction weights. Higher `alpha` means stronger positive attribute push; higher `beta` means stronger negative attribute push.

### Interpretation

The baseline is intentionally simple: it does not learn a transformation and does not use visual attribute directions.
The best run in the current sweep, `alpha=2.0` and `beta=1.0`, improves recall@10 over the default `alpha=1.0`, `beta=1.0`.

This suggests that, on this subset, a stronger positive text direction helps. The next natural improvement is to replace raw text directions with visual attribute directions, or to add a small gating mechanism that controls how strongly each attribute modifies the reference image.

These numbers are useful for development, but they are subset results. Final assignment numbers should be reported on the official full CelebA test split when the full test embeddings are available.

## 4.3. Visual-Direction Retrieval

After the CLIP text-arithmetic baseline, we tested a second retrieval variant based on visual attribute directions. For each CelebA attribute, the direction was computed from the train split as:

```text
direction(attribute) = normalize(mean(images where attribute = +1) - mean(images where attribute = -1))
```

The retrieval query then used the reference image plus signed visual directions:

```text
q = normalize(alpha * reference_image + beta_pos * sum(positive directions) - beta_neg * sum(negative directions))
```

This was implemented separately from the baseline in `src/retrieval/visual_direction_retrieval.py`, so the baseline results remain a fixed reference point.


### Visual-Direction Sweep

We evaluated a small fixed-weight grid using the same test subset, query file, and metrics as the baseline:

```text
alpha:    0.5, 1.0, 2.0
beta_pos: 0.5, 1.0, 2.0
beta_neg: 0.5, 1.0, 2.0
```

The sweep was run with `scripts/run_visual_direction_retrieval.py`, and the sorted results were saved to `outputs/visual_direction_run/summary.csv`.


In [ ]:
visual_summary_path = PROJECT_ROOT / "outputs/visual_direction_run/summary.csv"
visual_summary = pd.read_csv(visual_summary_path)
visual_summary.head(5)[[
    "alpha", "beta_pos", "beta_neg",
    "recall@1", "precision@1",
    "recall@5", "precision@5",
    "recall@10", "precision@10",
]]

,alpha,beta_pos,beta_neg,recall@1,precision@1,recall@5,precision@5,recall@10,precision@10
0,2.0,1.0,0.5,0.050610,0.050610,0.146341,0.038537,0.218293,0.032256
1,1.0,0.5,0.5,0.047561,0.047561,0.132317,0.035610,0.208537,0.030854
2,2.0,1.0,1.0,0.047561,0.047561,0.132317,0.035610,0.208537,0.030854
3,2.0,2.0,0.5,0.042683,0.042683,0.135366,0.034634,0.196341,0.029085
4,1.0,1.0,0.5,0.039024,0.039024,0.124390,0.032317,0.190244,0.027866


### Baseline vs Visual Directions

The best visual-direction setting was compared against the best baseline setting. The visual-direction run was close, but it did not improve over the CLIP text-arithmetic baseline.


In [12]:
best_baseline = summary.sort_values("recall@10", ascending=False).iloc[0]
best_visual = visual_summary.sort_values("recall@10", ascending=False).iloc[0]

comparison = pd.DataFrame([
    {
        "method": "CLIP text arithmetic baseline",
        "alpha": best_baseline["alpha"],
        "beta": best_baseline["beta"],
        "beta_pos": None,
        "beta_neg": None,
        "recall@1": best_baseline["recall@1"],
        "recall@5": best_baseline["recall@5"],
        "recall@10": best_baseline["recall@10"],
        "precision@10": best_baseline["precision@10"],
    },
    {
        "method": "visual directions",
        "alpha": best_visual["alpha"],
        "beta": None,
        "beta_pos": best_visual["beta_pos"],
        "beta_neg": best_visual["beta_neg"],
        "recall@1": best_visual["recall@1"],
        "recall@5": best_visual["recall@5"],
        "recall@10": best_visual["recall@10"],
        "precision@10": best_visual["precision@10"],
    },
])
comparison


,method,alpha,beta,beta_pos,beta_neg,recall@1,recall@5,recall@10,precision@10
0,CLIP text arithmetic baseline,2.0,1.0,NaN,NaN,0.056098,0.150000,0.226220,0.033232
1,visual directions,2.0,NaN,1.0,0.5,0.050610,0.146341,0.218293,0.032256


### Visual-Direction Takeaway

The best visual-direction setting was `alpha=2.0`, `beta_pos=1.0`, `beta_neg=0.5`, with `recall@1=0.0506`, `recall@5=0.1463`, and `recall@10=0.2183`. The best baseline setting was `alpha=2.0`, `beta=1.0`, with `recall@1=0.0561`, `recall@5=0.1500`, and `recall@10=0.2262`. Therefore, simple fixed visual directions were competitive but slightly worse than text directions. This suggests that raw visual directions alone may be noisy or too global, and the next improvement should probably combine text and visual directions or weight attributes based on difficulty/reliability. The strongest visual-direction runs consistently favored a smaller negative-direction weight, with the best setting using `beta_neg=0.5`; this suggests that negative visual directions are less trustworthy in this setup and may be more destructive than positive directions when they are applied too strongly.


## 4.4. Hybrid Text + Visual Fusion

Since visual directions alone may be noisy and may contain correlated CelebA attribute information, we tested a query-level hybrid that keeps the CLIP text direction as the main semantic signal and adds visual directions as a dataset-specific correction. The query is:

```text
q = normalize(
    alpha * reference_image_embedding
    + beta_text * text_delta
    + beta_visual_pos * visual_pos_delta
    - beta_visual_neg * visual_neg_delta
)
```

where `text_delta = sum(text_embeddings[pos_attrs]) - sum(text_embeddings[neg_attrs])`, `visual_pos_delta = sum(visual_directions[pos_attrs])`, and `visual_neg_delta = sum(visual_directions[neg_attrs])`. This was implemented in `src/retrieval/hybrid_fusion_retrieval.py` and run with `scripts/run_hybrid_fusion_retrieval.py`.


In [8]:
hybrid_summary_path = PROJECT_ROOT / "outputs/hybrid_fusion_run/summary.csv"
hybrid_summary = pd.read_csv(hybrid_summary_path)
best_hybrid = hybrid_summary.sort_values("recall@10", ascending=False).iloc[0]

hybrid_comparison = pd.DataFrame([
    {
        "method": "CLIP text arithmetic baseline",
        "alpha": best_baseline["alpha"],
        "beta_text": best_baseline["beta"],
        "beta_visual_pos": None,
        "beta_visual_neg": None,
        "recall@1": best_baseline["recall@1"],
        "recall@5": best_baseline["recall@5"],
        "recall@10": best_baseline["recall@10"],
        "precision@10": best_baseline["precision@10"],
    },
    {
        "method": "visual directions",
        "alpha": best_visual["alpha"],
        "beta_text": None,
        "beta_visual_pos": best_visual["beta_pos"],
        "beta_visual_neg": best_visual["beta_neg"],
        "recall@1": best_visual["recall@1"],
        "recall@5": best_visual["recall@5"],
        "recall@10": best_visual["recall@10"],
        "precision@10": best_visual["precision@10"],
    },
    {
        "method": "hybrid text + visual fusion",
        "alpha": best_hybrid["alpha"],
        "beta_text": best_hybrid["beta_text"],
        "beta_visual_pos": best_hybrid["beta_visual_pos"],
        "beta_visual_neg": best_hybrid["beta_visual_neg"],
        "recall@1": best_hybrid["recall@1"],
        "recall@5": best_hybrid["recall@5"],
        "recall@10": best_hybrid["recall@10"],
        "precision@10": best_hybrid["precision@10"],
    },
])

hybrid_comparison


NameError: name 'best_baseline' is not defined

### Hybrid Takeaway

The best hybrid setting was `alpha=1.0`, `beta_text=2.0`, `beta_visual_pos=0.5`, and `beta_visual_neg=0.25`, with `recall@1=0.0671`, `recall@5=0.1927`, `recall@10=0.2860`, and `precision@10=0.0413`. This improves over both the best text-only baseline (`recall@10=0.2262`) and the best visual-direction-only run (`recall@10=0.2183`). The smaller negative visual weight again supports the hypothesis that negative visual directions are less reliable than positive ones when used too strongly.

This parameter pattern is also informative. The text-only baseline needed a larger reference weight (`alpha=2.0`), while the hybrid works best with a smaller reference weight (`alpha=1.0`) and a stronger modification signal (`beta_text=2.0`). This suggests that, once the attribute modification is supported by both CLIP text embeddings and CelebA visual directions, the query can move farther away from the original reference image and rely more on the requested edit.

A natural next step is therefore a finer grid around this region:

```text
alpha:           0.5, 0.75, 1.0, 1.25, 1.5
beta_text:       1.5, 2.0, 2.5, 3.0
beta_visual_pos: 0.25, 0.5, 0.75, 1.0
beta_visual_neg: 0.0, 0.1, 0.25, 0.4, 0.5
```

The `beta_visual_neg=0.0` case is especially important: if negative visual directions are very noisy, the best hybrid may use visual directions only for positive attributes and leave negative edits to the CLIP text direction.



### 4.4.1. Prompt-Ensemble Hybrid Check

As a small follow-up, we replaced the single CLIP attribute prompt in the hybrid method with an average of multiple positive prompt templates per attribute. The retrieval formula stays the same; only the text embedding used inside `text_delta` changes.


In [ ]:
prompt_ensemble_summary_path = PROJECT_ROOT / "outputs/hybrid_prompt_ensemble_run/summary.csv"
prompt_ensemble_summary = pd.read_csv(prompt_ensemble_summary_path)
best_prompt_ensemble = prompt_ensemble_summary.sort_values("recall@10", ascending=False).iloc[0]

pd.DataFrame([
    {
        "method": "single-prompt hybrid",
        "recall@10": best_hybrid["recall@10"],
        "precision@10": best_hybrid["precision@10"],
    },
    {
        "method": "prompt-ensemble hybrid",
        "recall@10": best_prompt_ensemble["recall@10"],
        "precision@10": best_prompt_ensemble["precision@10"],
    },
])


### Prompt-Ensemble Takeaway

Prompt ensembling did not improve the main retrieval score in this experiment: the single-prompt hybrid reached `recall@10=0.2860`, while the prompt-ensemble hybrid reached `recall@10=0.2811`. This suggests that averaging several text prompts mostly smooths the text signal, but does not add a useful correction beyond what the single prompt and CelebA visual directions already provide.

The prompt-ensemble run has a slightly higher `precision@10` (`0.0416` vs `0.0413`), but the difference is very small, so it is better treated as a side observation rather than the main direction. The strongest method remains the hybrid with the original CLIP text prompt plus visual directions: CLIP text provides the semantic edit, and the CelebA visual directions provide the dataset-specific adjustment.


## 4.5. Reliability weighted hybrid text and visual fusion


### 4.5.1. Correlation-based reliability weighted hybrid text and visual fusion

# 5. Discussion and conclusion

# 6. References

Liang, V. W., Zhang, Y., Kwon, Y., Yeung, S., & Zou, J. Y. (2022). Mind the gap: Understanding the modality gap in multi-modal contrastive representation learning. Advances in Neural Information Processing Systems, 35, 17612-17625.

Radford, A., Kim, J. W., Hallacy, C., Ramesh, A., Goh, G., Agarwal, S., ... & Sutskever, I. (2021, July). Learning transferable visual models from natural language supervision. In International conference on machine learning (pp. 8748-8763). PmLR.

Patashnik, O., Wu, Z., Shechtman, E., Cohen-Or, D., & Lischinski, D. (2021). Styleclip: Text-driven manipulation of stylegan imagery. In Proceedings of the IEEE/CVF international conference on computer vision (pp. 2085-2094).